# 03 Nebula Analysis

This notebook covers full data analysis of a nebula FITS image brightness mapping, contour mapping, star detection, SNR analysis, and more.

**Object:** Any deep sky nebula or galaxy  
**Requirements:** astropy, numpy, matplotlib, photutils, scipy

In [ ]:
from astropy.io import fits
from astropy.visualization import AsinhStretch, PercentileInterval
from astropy.stats import sigma_clipped_stats
from photutils.detection import DAOStarFinder
from scipy.stats import gaussian_kde
from scipy.spatial.distance import cdist
import numpy as np
import matplotlib.pyplot as plt

fits_file = "your_image.fits"  # Change this to your file path

with fits.open(fits_file) as hdul:
    data   = hdul[0].data
    header = hdul[0].header

rgb = np.transpose(data, (1, 2, 0))

# Apply Asinh stretch
interval = PercentileInterval(99.5)
stretch  = AsinhStretch(a=0.01)  # Stretch strength lower = stronger (try 0.001 to 0.1)

rgb_new = np.zeros_like(rgb)
for i in range(3):
    vmin, vmax     = interval.get_limits(rgb[:,:,i])
    normalized     = np.clip((rgb[:,:,i] - vmin) / (vmax - vmin), 0, 1)
    rgb_new[:,:,i] = stretch(normalized)

## Brightness Map

Calculates the mean brightness across all three RGB channels and displays it as a heatmap.
Yellow/white = brightest regions. Black = darkest.

In [ ]:
brightness = np.mean(rgb_new, axis=2)

plt.figure(figsize=(10, 6))
plt.imshow(brightness, cmap='hot', origin='lower')
plt.colorbar(label='Brightness')
plt.title('Brightness Map')
plt.axis('off')
plt.show()

print(f'Mean brightness: {np.mean(brightness):.4f}')
print(f'Max brightness:  {np.max(brightness):.4f}')
print(f'Min brightness:  {np.min(brightness):.4f}')

## Contour Map

Draws lines connecting pixels of equal brightness like a topographic map.
Reveals the layered emission structure of the nebula.

In [ ]:
plt.figure(figsize=(10, 6))
plt.imshow(rgb_new, origin='lower')
plt.contour(brightness, levels=10, colors='white',  # Number of contour lines increase for more detail
            alpha=0.5, linewidths=0.5)
plt.title('Nebula Contours')
plt.axis('off')
plt.show()

## Star Detection

Uses DAOStarFinder a professional astronomy algorithm to automatically detect stars based on brightness and shape.

### How it works
 `sigma_clipped_stats` computes image statistics while ignoring outliers
 `fwhm` sets the expected star size in pixels — adjust based on your image
 `threshold` sets the minimum brightness for detection (5x the noise level)

In [ ]:
mean, median, std = sigma_clipped_stats(brightness)

daofind = DAOStarFinder(fwhm=3.0, threshold=5.*std)
sources = daofind(brightness - median)

print(f'Stars detected: {len(sources)}')

plt.figure(figsize=(10, 6))
plt.imshow(brightness, cmap='hot', origin='lower')
plt.scatter(sources['xcentroid'], sources['ycentroid'],
            s=20, facecolors='none', edgecolors='cyan',
            linewidths=0.8)
plt.title(f'Star Detection ({len(sources)} stars)')
plt.axis('off')
plt.show()

## Star Density Heatmap

Kernel Density Estimation (KDE) shows where stars are most concentrated.
Bright points = high star density = likely cluster location.

In [ ]:
x  = sources['xcentroid']
y  = sources['ycentroid']
xy = np.vstack([x, y])
kde = gaussian_kde(xy)
z   = kde(xy)

plt.figure(figsize=(10, 6))
plt.imshow(rgb_new, origin='lower', alpha=0.7)
plt.scatter(x, y, c=z, cmap='hot', s=40, alpha=0.9)
plt.colorbar(label='Star Density')
plt.title('Star Density Heatmap')
plt.axis('off')
plt.show()

## Signal-to-Noise Ratio (SNR) Map

Divides the image into regions and calculates SNR for each.
 Green = high SNR (clean sky regions)
 Red = low SNR (bright core or noisy areas)

The core always has lower SNR than the edges in emission nebulae this is physically expected.

In [ ]:
region_size = 200
results     = []

for y in range(0, brightness.shape[0] - region_size, region_size):
    for x in range(0, brightness.shape[1] - region_size, region_size):
        region = brightness[y:y+region_size, x:x+region_size]
        signal = np.mean(region)
        noise  = np.std(region)
        snr    = signal / noise if noise > 0 else 0
        results.append({'x': x, 'y': y, 'snr': snr})

snr_map = np.zeros(brightness.shape)
for r in results:
    snr_map[r['y']:r['y']+region_size,
            r['x']:r['x']+region_size] = r['snr']

plt.figure(figsize=(10, 6))
plt.imshow(snr_map, cmap='RdYlGn', origin='lower')
plt.colorbar(label='SNR')
plt.title('Signal-to-Noise Ratio Map')
plt.axis('off')
plt.show()